# Análisis exploratorio no supervisado del conteo FIRMS

Este notebook compara las clases candidatas actuales (`0`, `1`, `2–3`, `>=4`) con tres grupos obtenidos mediante K-Means. El clustering se ajusta exclusivamente sobre observaciones positivas del período de entrenamiento 2018–2023. Los ceros permanecen separados como `Sin detección`. No se modifica el target original ni se entrena un modelo predictivo.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

pd.set_option("display.max_columns", 30)

## Partición temporal y controles

Se utiliza 2018–2023 como `train`; 2024 y 2025 quedan fuera del ajuste para futuras etapas de validación y prueba. La fecha de inicio de semana define la partición.

In [ ]:
RUTA_DATOS = "data/processed/firms_open_meteo_weekly/open_meteo_firms_weekly_target_available.parquet"
TRAIN_INICIO = pd.Timestamp("2018-01-01")
TRAIN_FIN = pd.Timestamp("2023-12-31")

datos = pd.read_parquet(RUTA_DATOS)
datos["fecha_inicio_semana"] = pd.to_datetime(datos["fecha_inicio_semana"], errors="raise")
columnas_requeridas = {"fecha_inicio_semana", "cantidad_detecciones", "nivel_actividad_firms"}
assert columnas_requeridas.issubset(datos.columns)
assert datos[list(columnas_requeridas)].isna().sum().sum() == 0
assert datos["cantidad_detecciones"].ge(0).all()

train = datos.loc[datos["fecha_inicio_semana"].between(TRAIN_INICIO, TRAIN_FIN)].copy()
train_positivos = train.loc[train["cantidad_detecciones"].gt(0)].copy()

pd.DataFrame({
    "particion": ["train completo", "train positivo", "ceros separados"],
    "observaciones": [len(train), len(train_positivos), train["cantidad_detecciones"].eq(0).sum()]
})

In [ ]:
X = np.log1p(train_positivos[["cantidad_detecciones"]].astype(float))
kmeans = KMeans(n_clusters=3, random_state=42, n_init=20)
train_positivos["cluster_original"] = kmeans.fit_predict(X)

orden_clusters = np.argsort(kmeans.cluster_centers_.ravel())
nombres_ordenados = ["Bajo", "Moderado", "Alto"]
mapa_clusters = {int(cluster): nombre for cluster, nombre in zip(orden_clusters, nombres_ordenados)}
train_positivos["nivel_actividad_kmeans"] = train_positivos["cluster_original"].map(mapa_clusters)
assert train_positivos["nivel_actividad_kmeans"].notna().all()

## Centroides y resumen de los clusters

Los centroides se transforman nuevamente con `expm1`. Al no ser necesariamente enteros, representan el centro geométrico del conteo dentro de cada cluster, no un punto de corte.

In [ ]:
centroides = pd.DataFrame([
    {"cluster": mapa_clusters[int(cluster)],
     "centroide_log1p": float(kmeans.cluster_centers_[cluster, 0]),
     "centroide_escala_original": float(np.expm1(kmeans.cluster_centers_[cluster, 0]))}
    for cluster in orden_clusters
]).set_index("cluster").reindex(nombres_ordenados)
centroides.round(4)

In [ ]:
resumen_clusters = (
    train_positivos.groupby("nivel_actividad_kmeans")["cantidad_detecciones"]
    .agg(["count", "min", "max", "mean", "median"])
    .reindex(nombres_ordenados)
)
resumen_clusters.round(3)

## Valores asignados a cada cluster

In [ ]:
valores_por_cluster = (
    train_positivos.groupby("nivel_actividad_kmeans")["cantidad_detecciones"]
    .apply(lambda serie: sorted(int(valor) for valor in serie.unique()))
    .reindex(nombres_ordenados)
    .rename("valores_cantidad_detecciones")
)
valores_por_cluster.to_frame()

## Comparación con las clases actuales

La tabla se limita a positivos, por lo que `Sin detección` no participa del ajuste y permanece conceptualmente separada.

In [ ]:
orden_actual_positivo = ["Bajo", "Moderado", "Alto"]
tabla_cruzada = pd.crosstab(
    train_positivos["nivel_actividad_firms"].astype("string"),
    train_positivos["nivel_actividad_kmeans"],
    rownames=["Clase FIRMS actual"], colnames=["Cluster K-Means"],
).reindex(index=orden_actual_positivo, columns=nombres_ordenados, fill_value=0)
tabla_cruzada

In [ ]:
comparacion_cortes = pd.DataFrame({
    "clase": ["Sin detección", "Bajo", "Moderado", "Alto"],
    "cortes_actuales": ["0", "1", "2–3", ">=4"],
    "asignacion_kmeans_train": ["0 (separado)", "1", "2–4", ">=5 observado"],
})
comparacion_cortes

## Visualizaciones de los clusters

Los gráficos utilizan únicamente las observaciones positivas de `train`. La escala logarítmica permite mostrar los valores extremos sin ocultar la concentración en conteos pequeños.

In [ ]:
colores_clusters = {"Bajo": "#2ca02c", "Moderado": "#ffbf00", "Alto": "#d62728"}
frecuencias_cluster = (
    train_positivos.groupby(["nivel_actividad_kmeans", "cantidad_detecciones"])
    .size().rename("frecuencia").reset_index()
)

fig, ax = plt.subplots(figsize=(12, 5))
for cluster in nombres_ordenados:
    parte = frecuencias_cluster.loc[frecuencias_cluster["nivel_actividad_kmeans"].eq(cluster)]
    ax.bar(parte["cantidad_detecciones"], parte["frecuencia"],
           color=colores_clusters[cluster], alpha=0.8, label=cluster)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xticks([1, 2, 3, 4, 5, 10, 20, 50, 100, 173])
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.set_title("Frecuencia de conteos FIRMS positivos por cluster — train 2018–2023")
ax.set_xlabel("Cantidad de detecciones FIRMS por departamento-semana (escala log)")
ax.set_ylabel("Número de observaciones (escala log)")
ax.legend(title="Cluster K-Means")
ax.grid(alpha=0.25, which="both")
plt.tight_layout()
plt.show()

In [ ]:
valores_boxplot = [
    train_positivos.loc[train_positivos["nivel_actividad_kmeans"].eq(cluster), "cantidad_detecciones"].astype(float)
    for cluster in nombres_ordenados
]
fig, ax = plt.subplots(figsize=(9, 5))
box = ax.boxplot(valores_boxplot, tick_labels=nombres_ordenados, patch_artist=True, showfliers=True)
for patch, cluster in zip(box["boxes"], nombres_ordenados):
    patch.set_facecolor(colores_clusters[cluster])
    patch.set_alpha(0.75)
ax.set_yscale("log")
ax.set_title("Distribución de detecciones FIRMS positivas por cluster — train 2018–2023")
ax.set_xlabel("Cluster K-Means ordenado por centroide")
ax.set_ylabel("Cantidad de detecciones por departamento-semana (escala log)")
ax.grid(axis="y", alpha=0.25, which="both")
plt.tight_layout()
plt.show()

In [ ]:
matriz = tabla_cruzada.to_numpy()
fig, ax = plt.subplots(figsize=(8, 5))
imagen = ax.imshow(matriz, cmap="Blues", aspect="auto")
ax.set_xticks(range(len(nombres_ordenados)), labels=nombres_ordenados)
ax.set_yticks(range(len(orden_actual_positivo)), labels=orden_actual_positivo)
ax.set_title("Clases FIRMS actuales frente a clusters K-Means — train 2018–2023")
ax.set_xlabel("Cluster K-Means")
ax.set_ylabel("Clase FIRMS actual")
for fila in range(matriz.shape[0]):
    for columna in range(matriz.shape[1]):
        ax.text(columna, fila, f"{matriz[fila, columna]:,}",
                ha="center", va="center",
                color="white" if matriz[fila, columna] > matriz.max() / 2 else "black")
fig.colorbar(imagen, ax=ax, label="Número de observaciones positivas")
plt.tight_layout()
plt.show()

## Conclusión

K-Means respalda separar `1` como Bajo y agrupar `2–3` como Moderado, pero ubica también el valor `4` en Moderado y comienza Alto en `5`. Esto afecta únicamente a las observaciones con exactamente cuatro detecciones: no contradice la estructura general, pero propone revisar el límite Moderado/Alto. Como el corte actual `>=4` conserva más observaciones en Alto y fue definido antes de este análisis, conviene mantenerlo provisionalmente para no redefinir el target a partir de un único ajuste. La alternativa `2–4 / >=5` debería compararse posteriormente por estabilidad entre períodos y departamentos usando solamente entrenamiento; este notebook no modifica la definición vigente.